# 🏥 Obesity Dataset — Exploratory Data Analysis
**Dataset:** `ObesityDataSet_raw_and_data_sinthetic.csv`  
**Target:** `NObeyesdad` — 7-class obesity level  
**Rows:** 2,111 | **Features:** 16 (8 numerical, 8 categorical)

---
### Table of Contents
1. [Setup & Load Data](#1)
2. [Basic Info & Summary Statistics](#2)
3. [Missing Values](#3)
4. [Target Variable Distribution](#4)
5. [Numerical Feature Distributions](#5)
6. [BMI Engineering](#6)
7. [Correlation Matrix (raw)](#7)
8. [Numerical Features vs Target](#8)
9. [Categorical Features vs Target](#9)
10. [Lifestyle Features](#10)
11. [Pairplot](#11)
12. [Feature Importance Summary](#12)
13. [Feature Encoding](#13)
14. [Feature Selection & Why](#14)
15. [Outlier Analysis](#15)
16. [Interaction Features](#16)
17. [Correlation Matrix (encoded)](#17)
18. [Random Forest Feature Importance](#18)
19. [Class Imbalance & Class Weights](#19)

## 1. Setup & Load Data <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────────────────────
PALETTE   = "Set2"
FIG_W     = 10          # default figure width
FIG_H     = 5           # default figure height
TITLE_FS  = 14
LABEL_FS  = 11
ROT       = 45

sns.set_theme(style="whitegrid", palette=PALETTE, font_scale=1.1)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize":  TITLE_FS,
    "axes.labelsize":  LABEL_FS,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})
pd.set_option('display.max_columns', None)

# Ordered obesity levels for consistent plots
OB_ORDER = [
    "Insufficient_Weight", "Normal_Weight",
    "Overweight_Level_I",  "Overweight_Level_II",
    "Obesity_Type_I",      "Obesity_Type_II", "Obesity_Type_III"
]

In [ ]:
df = pd.read_csv("ObesityDataSet_raw_and_data_sinthetic.csv")
df['NObeyesdad'] = pd.Categorical(df['NObeyesdad'], categories=OB_ORDER, ordered=True)
df.head()

## 2. Basic Info & Summary Statistics <a id='2'></a>
- **2,111 rows, 17 columns** — no missing values.
- 8 float features (continuous/ordinal), 9 object features (binary or multi-category).
- Age is right-skewed (14–61, median ~23). Weight spans 39–173 kg.

In [ ]:
print(f"Shape: {df.shape}")
print()
df.info()

In [ ]:
# Separate numeric / categorical stats for readability
cat_cols = df.select_dtypes(include='object').columns.tolist()
num_cols = df.select_dtypes(exclude='object').columns.tolist()
if 'BMI' in num_cols: num_cols.remove('BMI')   # added later

print("Numerical summary:")
display(df[num_cols].describe().round(2))
print("\nCategorical summary:")
display(df[cat_cols].describe())

## 3. Missing Values <a id='3'></a>
> **No missing values detected** — the dataset is complete and ready to use.

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "None — dataset is clean ✓")

## 4. Target Variable Distribution <a id='4'></a>
The target is reasonably balanced across 7 classes (~270–360 samples each).  
*Obesity Type I* is the most frequent category.

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
counts = df['NObeyesdad'].value_counts()[OB_ORDER]
bars = ax.bar(OB_ORDER, counts.values,
              color=sns.color_palette(PALETTE, len(OB_ORDER)))
ax.bar_label(bars, padding=3, fontsize=10)
ax.set_title("Target Variable — Obesity Level Distribution")
ax.set_xlabel("Obesity Level")
ax.set_ylabel("Count")
ax.set_xticklabels(OB_ORDER, rotation=ROT, ha='right')
plt.tight_layout()
plt.show()

## 5. Numerical Feature Distributions <a id='5'></a>
- **Age** — young-adult skewed (most 18–30).
- **Height / Weight** — roughly normal; Weight shows a right tail.
- **FCVC, NCP, CH2O** — discrete pseudo-continuous scales.
- **FAF** — heavily right-skewed; many people report zero physical activity.
- **TUE** — most values clustered near 0–1 hours.

In [ ]:
n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols   # ceil division

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 4))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=25, color=sns.color_palette(PALETTE)[i % 6],
                 edgecolor='white', linewidth=0.5)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frequency")

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numerical Feature Distributions", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 6. BMI Feature Engineering <a id='6'></a>
> **BMI = Weight / Height²**  — a derived feature that directly encodes the obesity definition.  
> As expected, BMI almost perfectly separates obesity classes. It will be a **top predictor**.

In [ ]:
df['BMI'] = df['Weight'] / (df['Height'] ** 2)

fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
sns.boxplot(data=df, x='NObeyesdad', y='BMI', order=OB_ORDER,
            palette=PALETTE, ax=ax)
ax.set_title("BMI vs Obesity Level")
ax.set_xlabel("Obesity Level")
ax.set_ylabel("BMI (kg/m²)")
ax.set_xticklabels(OB_ORDER, rotation=ROT, ha='right')
plt.tight_layout()
plt.show()

## 7. Correlation Matrix <a id='7'></a>
- **Weight ↔ BMI** (0.93): near perfect — expected.
- **Weight ↔ Height** (0.47): taller people tend to weigh more.
- **FAF ↔ TUE** (−0.16): active people spend slightly less screen time.
- Other correlations are weak, suggesting low multicollinearity among lifestyle features.

In [ ]:
num_all = df.select_dtypes(include='number').columns.tolist()
corr = df[num_all].corr()

fig, ax = plt.subplots(figsize=(FIG_W, FIG_W))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Correlation Matrix — Raw Features (lower triangle)", pad=12)
plt.tight_layout()
plt.show()

## 8. Numerical Features vs Target <a id='8'></a>
Boxplots show how each numeric feature separates across obesity classes.  
**Key signals:**
- `Weight` and `BMI` — monotonically increasing ✅ **(strongest)**
- `FAF` (physical activity) — higher in normal/underweight ✅
- `Age` — moderate separation
- `Height`, `FCVC`, `NCP`, `CH2O`, `TUE` — weak but present

In [ ]:
plot_cols = ['Weight', 'BMI', 'FAF', 'Age', 'Height', 'FCVC', 'NCP', 'CH2O', 'TUE']
n = len(plot_cols)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(18, 4 * ((n + 2) // 3)))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    sns.boxplot(data=df, x='NObeyesdad', y=col, order=OB_ORDER,
                palette=PALETTE, ax=axes[i])
    axes[i].set_title(f"{col} vs Obesity Level")
    axes[i].set_xlabel("")
    axes[i].set_ylabel(col)
    axes[i].set_xticklabels(OB_ORDER, rotation=ROT, ha='right', fontsize=8)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Numerical Features vs Obesity Level", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 9. Categorical Features vs Target <a id='9'></a>
- `family_history_with_overweight` — strongest binary signal; "yes" associates heavily with obesity.
- `FAVC` (frequent high-cal food) — "yes" skews toward obesity classes.
- `CAEC` (eating between meals) — "Sometimes" is most common; "Always" correlates with obesity.
- `SMOKE`, `SCC` — very imbalanced (nearly all "no"), limited signal.
- `MTRANS` (transport) — Public transport users dominate, slight differences across classes.

In [ ]:
plot_cat = [c for c in cat_cols if c != 'NObeyesdad']
n = len(plot_cat)
fig, axes = plt.subplots((n + 1) // 2, 2, figsize=(16, 4 * ((n + 1) // 2)))
axes = axes.flatten()

for i, col in enumerate(plot_cat):
    sns.countplot(data=df, x=col, hue='NObeyesdad', hue_order=OB_ORDER,
                  palette=PALETTE, ax=axes[i])
    axes[i].set_title(f"{col} vs Obesity Level")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=ROT, ha='right')
    axes[i].legend(title="Obesity Level", bbox_to_anchor=(1.01, 1),
                   fontsize=7, title_fontsize=8)

for j in range(n, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Categorical Features vs Obesity Level", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 10. Lifestyle Features Deep Dive <a id='10'></a>
Violin plots reveal distribution shape, not just quartiles.

In [ ]:
lifestyle_num = ['FAF', 'TUE', 'CH2O']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, lifestyle_num):
    sns.violinplot(data=df, x='NObeyesdad', y=col, order=OB_ORDER,
                   palette=PALETTE, inner='box', ax=ax)
    ax.set_title(f"{col} Distribution by Obesity Level")
    ax.set_xlabel("")
    ax.set_ylabel(col)
    ax.set_xticklabels(OB_ORDER, rotation=ROT, ha='right', fontsize=8)

plt.suptitle("Lifestyle Numerical Features (Violin)", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 11. Pairplot (sampled) <a id='11'></a>
Only key numeric features sampled to 500 rows for performance.

In [ ]:
key_num = ['Weight', 'BMI', 'Age', 'FAF', 'TUE']
sample_df = df.sample(n=500, random_state=42)

g = sns.pairplot(sample_df[key_num + ['NObeyesdad']],
                 hue='NObeyesdad', hue_order=OB_ORDER,
                 palette=PALETTE, diag_kind='kde',
                 plot_kws={'alpha': 0.5, 's': 20})
g.figure.suptitle("Pairplot — Key Numerical Features (n=500)", y=1.02, fontsize=14)
plt.show()

## 12. Feature Importance Summary <a id='12'></a>

| Priority | Feature | Type | Insight |
|----------|---------|------|---------|
| ⭐⭐⭐ | **BMI** | Engineered | Near-perfect monotonic separation across all 7 classes |
| ⭐⭐⭐ | **Weight** | Numerical | Strongest raw signal; corr 0.93 with BMI |
| ⭐⭐ | **family_history_with_overweight** | Categorical | Genetic predisposition — highest binary signal |
| ⭐⭐ | **FAF** (Physical Activity) | Numerical | Inverse relationship with obesity severity |
| ⭐⭐ | **FAVC** (High-cal food) | Categorical | "yes" strongly linked to higher obesity levels |
| ⭐ | **Age** | Numerical | Moderate separation; obese individuals slightly older |
| ⭐ | **FCVC** (Vegetable consumption) | Numerical | Higher in healthier classes |
| ⭐ | **CAEC** (Eating between meals) | Ordinal | "Always" correlates with obesity |
| ⭐ | **TUE** (Screen time) | Numerical | Subtle lifestyle indicator |
| ➖ | **SMOKE**, **SCC** | Categorical | Near-constant — very limited predictive value |

> The sections below encode the data, validate this ranking with Random Forest Gini importance, and prepare a model-ready feature set.

## 13. Feature Encoding <a id='13'></a>

We apply three different encoding strategies based on each feature's **semantic type**:

| Strategy | Features | Reason |
|----------|----------|--------|
| **Binary (0/1)** | `Gender`, `family_history_with_overweight`, `FAVC`, `SMOKE`, `SCC` | Two-value categoricals — a single bit is sufficient |
| **Ordinal integer** | `CAEC`, `CALC` | Frequency-ordered categories — preserving order is critical signal |
| **One-hot** | `MTRANS` | Nominal transport mode — no natural order |
| **Label encode** | `NObeyesdad` (target) | Ordered integer for modelling; mapping printed below |

> ⚠️ **BMI leakage note:** BMI is algebraically derived from `Weight` and `Height` — both already in the dataset.  
> It's included here for completeness, but consider **dropping BMI** when training to avoid encoding the answer.

In [ ]:
# ── Work on a clean copy so EDA cells above remain valid ────────────────────
df_enc = df.copy()

# 1. Binary encoding
binary_cols = ['Gender', 'family_history_with_overweight', 'FAVC', 'SMOKE', 'SCC']
binary_map  = {'yes': 1, 'no': 0, 'Male': 1, 'Female': 0}
for col in binary_cols:
    df_enc[col] = df_enc[col].map(binary_map)

# 2. Ordinal encoding — frequency/intensity order
freq_map = {'no': 0, 'Sometimes': 1, 'Frequently': 2, 'Always': 3}
df_enc['CAEC'] = df_enc['CAEC'].map(freq_map)
df_enc['CALC'] = df_enc['CALC'].map(freq_map)

# 3. One-hot for nominal transport
df_enc = pd.get_dummies(df_enc, columns=['MTRANS'], drop_first=True)

# 4. Label-encode target (was pd.Categorical — convert to int via cat.codes)
df_enc['NObeyesdad'] = df_enc['NObeyesdad'].cat.codes   # preserves OB_ORDER ranking

label_mapping = {name: code for code, name in enumerate(OB_ORDER)}
print("Target label mapping:")
for k, v in label_mapping.items():
    print(f"  {v}  →  {k}")

print(f"\nEncoded shape: {df_enc.shape}")
df_enc.head()

## 14. Feature Selection & Why <a id='14'></a>

Based on EDA + encoding, we drop features with **near-zero variance** or **redundancy**:

| Feature | Action | Reason |
|---------|--------|--------|
| `SMOKE` | ❌ Drop | ~98% "no" — near-zero variance, no signal |
| `SCC` | ❌ Drop | ~95% "no" — same issue |
| `BMI` | ⚠️ Optional drop | Derived from `Weight` + `Height` (data leakage risk) |
| `Height` | ⚠️ Consider | Once BMI is included, Height adds little independent signal |

**Kept features** are selected because they showed visible class separation in the EDA boxplots/countplots above.

In [ ]:
# Drop near-zero variance features
drop_cols = ['SMOKE', 'SCC']
df_model = df_enc.drop(columns=drop_cols)

# Separate features / target
TARGET = 'NObeyesdad'
X = df_model.drop(columns=[TARGET])
y = df_model[TARGET]

print("Selected features:", X.columns.tolist())
print(f"\nX shape: {X.shape} | y shape: {y.shape}")

## 15. Outlier Analysis <a id='15'></a>

`Age` has a small cluster of 50+ records that sit well outside the main 18–30 distribution.  
We visualise and flag them — they're kept (not removed) but worth watching in model evaluation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box + strip for Age
sns.boxplot(y=df['Age'], ax=axes[0], color=sns.color_palette(PALETTE)[0])
sns.stripplot(y=df['Age'], ax=axes[0], color='black', alpha=0.15, size=2)
axes[0].set_title("Age — Distribution & Outliers")
axes[0].set_ylabel("Age")

# Age coloured by obesity class
sns.boxplot(data=df, x='NObeyesdad', y='Age', order=OB_ORDER,
            palette=PALETTE, ax=axes[1])
axes[1].set_xticklabels(OB_ORDER, rotation=ROT, ha='right')
axes[1].set_title("Age by Obesity Level")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

# Flag high-age rows
age_outliers = df[df['Age'] > 50]
print(f"Records with Age > 50: {len(age_outliers)}")
print(age_outliers[['Age', 'Weight', 'NObeyesdad']].value_counts('NObeyesdad').to_string())

## 16. Interaction Features <a id='16'></a>

Non-linear interactions can capture compound effects invisible in individual features:

| Feature | Formula | Hypothesis |
|---------|---------|------------|
| `Weight_x_FAF` | `Weight × FAF` | High weight + low activity → strong obesity signal |
| `Age_x_FCVC` | `Age × FCVC` | Older people eating more vegetables may break obesity pattern |
| `BMI_x_FAF` | `BMI × FAF` | Physical activity moderating BMI effect |

In [ ]:
X['Weight_x_FAF'] = X['Weight'] * X['FAF']
X['Age_x_FCVC']   = X['Age']    * X['FCVC']
X['BMI_x_FAF']    = X['BMI']    * X['FAF']

print("Features after interactions:", X.shape[1])

# Quick correlation of new features with target
new_feats = ['Weight_x_FAF', 'Age_x_FCVC', 'BMI_x_FAF']
corr_with_target = pd.Series(
    {f: X[f].corr(y.astype(float)) for f in new_feats},
    name="corr_with_target"
).sort_values(ascending=False)
print("\nCorrelation of interaction features with target:")
print(corr_with_target.round(3))

## 17. Correlation Matrix — Encoded Features <a id='17'></a>

After encoding, we can see the true linear relationships between **all** features and the target.

In [ ]:
corr_enc = X.join(y).corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_enc, dtype=bool))
sns.heatmap(corr_enc, mask=mask, cmap="coolwarm", center=0,
            linewidths=0.3, annot=False, ax=ax)
ax.set_title("Correlation Matrix — Fully Encoded Features", fontsize=14, pad=12)
plt.tight_layout()
plt.show()

# Top correlations with target
print("Top features correlated with NObeyesdad (encoded):")
print(corr_enc['NObeyesdad'].drop('NObeyesdad').abs()
      .sort_values(ascending=False).round(3).to_string())

## 18. Random Forest Feature Importance <a id='18'></a>

A quick Random Forest validates (or overturns) the EDA-derived importance ranking using **Gini impurity** scores.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importance_df = (pd.DataFrame({'Feature': X.columns,
                                'Importance': rf.feature_importances_})
                   .sort_values('Importance', ascending=False)
                   .reset_index(drop=True))

print(f"RF Accuracy (quick check): {rf.score(X_test, y_test):.3f}")
print()

fig, ax = plt.subplots(figsize=(10, 7))
colors = sns.color_palette(PALETTE, len(importance_df))
bars = ax.barh(importance_df['Feature'][::-1],
               importance_df['Importance'][::-1],
               color=colors[::-1])
ax.set_title("Random Forest — Gini Feature Importances", fontsize=TITLE_FS)
ax.set_xlabel("Importance")
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
plt.tight_layout()
plt.show()

## 19. Class Imbalance & Class Weights <a id='19'></a>

The dataset is **roughly balanced** but not perfectly so. We compare two RF models:
- **Default weights** — treats all classes equally
- **Balanced weights** — automatically scales weights inversely proportional to class frequency  

Balanced weighting is generally safer when even slight imbalance exists.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.gridspec as gridspec

rf_default  = RandomForestClassifier(n_estimators=200, random_state=42,
                                      class_weight=None, n_jobs=-1)
rf_balanced = RandomForestClassifier(n_estimators=200, random_state=42,
                                      class_weight='balanced', n_jobs=-1)

rf_default.fit(X_train, y_train)
rf_balanced.fit(X_train, y_train)

acc_def = rf_default.score(X_test, y_test)
acc_bal = rf_balanced.score(X_test, y_test)

print(f"Accuracy — Default weights : {acc_def:.4f}")
print(f"Accuracy — Balanced weights: {acc_bal:.4f}")
print()

# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
short_labels = ['Insuf.', 'Normal', 'OW_I', 'OW_II', 'Ob_I', 'Ob_II', 'Ob_III']

for ax, model, title in zip(axes,
                             [rf_default, rf_balanced],
                             ['Default Weights', 'Balanced Weights']):
    ConfusionMatrixDisplay.from_estimator(
        model, X_test, y_test,
        display_labels=short_labels,
        cmap='Blues', colorbar=False, ax=ax)
    ax.set_title(f"Confusion Matrix — {title}\nAcc: {model.score(X_test, y_test):.4f}",
                 fontsize=12)
    ax.set_xticklabels(short_labels, rotation=ROT, ha='right')

plt.tight_layout()
plt.show()

# Per-class report comparison
print("\n── Default weights ──")
print(classification_report(y_test, rf_default.predict(X_test),
                             target_names=OB_ORDER))
print("── Balanced weights ──")
print(classification_report(y_test, rf_balanced.predict(X_test),
                             target_names=OB_ORDER))

# Testing BMI

In [ ]:
# Experiment A — "Realistic" model (no leakage)
X_real = X.drop(columns=['BMI', 'Weight', 'Height', 'Weight_x_FAF', 'BMI_x_FAF'])

# Experiment B — Full model (includes BMI)
X_full = X.copy()

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(rf_balanced, X, y, cv=cv, scoring='f1_macro')
print(f"CV F1-macro: {scores.mean():.4f} ± {scores.std():.4f}")

##  Leakage experiment — realistic vs full model

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Experiment A — no BMI/Weight/Height (realistic, lifestyle-only)
X_real = X.drop(columns=['BMI', 'Weight', 'Height', 'Weight_x_FAF', 'BMI_x_FAF'])

# Experiment B — full (includes BMI)
X_full = X.copy()

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                             random_state=42, n_jobs=-1)

score_real = cross_val_score(rf, X_real, y, cv=cv, scoring='f1_macro')
score_full = cross_val_score(rf, X_full, y, cv=cv, scoring='f1_macro')

print(f"Realistic (no BMI/Weight): {score_real.mean():.4f} ± {score_real.std():.4f}")
print(f"Full (with BMI):           {score_full.mean():.4f} ± {score_full.std():.4f}")
print(f"Leakage gap:               {score_full.mean() - score_real.mean():.4f}")

- Compares the leaky (exact BMI) configuration against the leakage-free one to make the shortcut explicit.

## Model comparison on both X_real and X_full

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, class_weight='balanced',
                                   random_state=42))
    ]),
    "Random Forest":       RandomForestClassifier(n_estimators=200,
                                                   class_weight='balanced',
                                                   random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200,
                                                       random_state=42),
}

results = {}
for name, model in models.items():
    s_real = cross_val_score(model, X_real, y, cv=cv, scoring='f1_macro')
    s_full = cross_val_score(model, X_full, y, cv=cv, scoring='f1_macro')
    results[name] = {'real': s_real.mean(), 'full': s_full.mean()}
    print(f"{name:25s}  real={s_real.mean():.4f}±{s_real.std():.4f}  "
          f"full={s_full.mean():.4f}±{s_full.std():.4f}")

# Bar chart comparison
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(results))
w = 0.35
ax.bar(x - w/2, [v['real'] for v in results.values()], w, label='Realistic (no BMI)')
ax.bar(x + w/2, [v['full'] for v in results.values()], w, label='Full (with BMI)')
ax.set_xticks(x)
ax.set_xticklabels(results.keys())
ax.set_ylabel("F1-macro (CV)")
ax.set_ylim(0.5, 1.02)
ax.set_title("Model Comparison — Realistic vs Full Feature Set")
ax.legend()
ax.bar_label(ax.containers[0], fmt='%.3f', padding=3)
ax.bar_label(ax.containers[1], fmt='%.3f', padding=3)
plt.tight_layout()
plt.show()

- Random Forest on the realistic (no exact BMI) features is the baseline.
- Gradient Boosting and Logistic Regression are compared alongside it; the comparison motivates a non-linear model.

## Hyperparameter tuning on the realistic model

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':      [100, 200, 400],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.5],
    'class_weight':      ['balanced', None],
}

rf_tune = RandomForestClassifier(random_state=42, n_jobs=-1)
search = RandomizedSearchCV(rf_tune, param_dist, n_iter=30, cv=cv,
                             scoring='f1_macro', random_state=42, n_jobs=-1,
                             verbose=1)
search.fit(X_real, y)

print(f"Best F1-macro: {search.best_score_:.4f}")
print(f"Best params:   {search.best_params_}")
best_rf = search.best_estimator_

##  SHAP — understand why the model predicts what it does

- FCVC, Gender, Age, and family_history are the top drivers — this is the genuinely interesting result. 
- Age_x_FCVC at 5th place validates that interaction feature. 
- FAVC is surprisingly weak (near bottom), which contradicts the EDA countplot signal.


In [ ]:
print(type(shap_values))
print(np.array(shap_values).shape)

In [ ]:
shap_arr = np.array(shap_values)

if shap_arr.ndim == 3 and shap_arr.shape[0] == 7:
    # Old format: (n_classes, n_samples, n_features) — index by class first
    shap.summary_plot(shap_values, X_test_r, class_names=OB_ORDER, plot_type="bar")
    shap.summary_plot(shap_values[6], X_test_r)          # class 6 = Obesity_Type_III

elif shap_arr.ndim == 3 and shap_arr.shape[2] == 7:
    # New format: (n_samples, n_features, n_classes) — transpose to old format
    shap_values_T = [shap_arr[:, :, i] for i in range(7)]
    shap.summary_plot(shap_values_T, X_test_r, class_names=OB_ORDER, plot_type="bar")
    shap.summary_plot(shap_values_T[6], X_test_r)        # class 6 = Obesity_Type_III

else:
    print("Unexpected shape:", shap_arr.shape)

- The diagonal (self-interactions) dominates for all features, meaning each feature contributes mostly independently. 
- The Weight × family_history off-diagonal is the only notable interaction — everything else is near zero.

# XGBOOST again

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                    use_label_encoder=False, eval_metric='mlogloss',
                    random_state=42, n_jobs=-1)

scores_xgb = cross_val_score(xgb, X_real, y, cv=cv, scoring='f1_macro')
print(f"XGBoost realistic: {scores_xgb.mean():.4f} ± {scores_xgb.std():.4f}")

# why FAVC is weak despite strong EDA signal

In [ ]:
# Partial dependence: FAVC effect controlling for family_history
from sklearn.inspection import PartialDependenceDisplay

fig, ax = plt.subplots(figsize=(10, 4))
PartialDependenceDisplay.from_estimator(
    best_rf, X_real,
    features=['FAVC', 'family_history_with_overweight', ('FAVC', 'family_history_with_overweight')],
    target=4,   # Obesity_Type_I
    ax=ax
)
plt.tight_layout()
plt.show()

# Per-class confusion to see which classes drive the errors:

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

y_pred_r = best_rf.predict(X_test_r)
ConfusionMatrixDisplay.from_predictions(
    y_test_r, y_pred_r,
    display_labels=[o.replace('_', '\n') for o in OB_ORDER],
    cmap='Blues', colorbar=False
)
plt.title("Best RF — Realistic Features\nConfusion Matrix")
plt.tight_layout()
plt.show()

# Checking class imbalance, to see why our model bleeds in the middle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

# ── 1. Raw counts & proportions ───────────────────────────────────────────────
counts = pd.Series(Counter(y)).reindex(range(7))
counts.index = OB_ORDER
props  = counts / counts.sum()

print("Class distribution:")
print(pd.DataFrame({'count': counts, 'proportion': props.round(4),
                    '%': (props * 100).round(2)}).to_string())

# Imbalance ratio: max_class / min_class  (>3 = problematic, >10 = severe)
imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")
print("→ Mild (<3) | Moderate (3–10) | Severe (>10)")

# ── 2. Visual ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar
bars = axes[0].bar(OB_ORDER, counts.values,
                   color=sns.color_palette("Set2", 7))
axes[0].bar_label(bars, padding=3)
axes[0].set_xticklabels(OB_ORDER, rotation=45, ha='right')
axes[0].set_title("Class Counts")
axes[0].set_ylabel("Count")
axes[0].axhline(counts.mean(), color='red', linestyle='--',
                label=f'Mean ({counts.mean():.0f})')
axes[0].legend()

# Proportion pie
axes[1].pie(counts.values, labels=OB_ORDER, autopct='%1.1f%%',
            colors=sns.color_palette("Set2", 7), startangle=90)
axes[1].set_title("Class Proportions")

plt.suptitle("Class Imbalance Analysis", fontsize=14)
plt.tight_layout()
plt.show()

# ── 3. Compute sklearn class weights ─────────────────────────────────────────
class_weights = compute_class_weight('balanced', classes=np.arange(7), y=y)
weight_df = pd.DataFrame({
    'class':  OB_ORDER,
    'count':  counts.values,
    'weight': class_weights.round(4)
}).set_index('class')
print("\nBalanced class weights (sklearn):")
print(weight_df.to_string())

# Strategy

In [ ]:
# ── 4. Strategy selector ──────────────────────────────────────────────────────

if imbalance_ratio < 3:
    print("""
    MILD IMBALANCE → class_weight='balanced' is sufficient.
    You're already doing this with rf_balanced. No resampling needed.
    """)

elif 3 <= imbalance_ratio <= 10:
    print("MODERATE → Try SMOTE oversampling on training set only.")

    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    from sklearn.ensemble import RandomForestClassifier

    # IMPORTANT: fit SMOTE only inside CV folds to avoid data leakage
    smote_rf = ImbPipeline([
        ('smote', SMOTE(random_state=42, k_neighbors=5)),
        ('clf',   RandomForestClassifier(n_estimators=200,
                                          random_state=42, n_jobs=-1))
    ])

    scores_smote = cross_val_score(smote_rf, X_real, y,
                                   cv=cv, scoring='f1_macro')
    scores_base  = cross_val_score(best_rf,  X_real, y,
                                   cv=cv, scoring='f1_macro')

    print(f"Base RF (balanced weights): {scores_base.mean():.4f} ± {scores_base.std():.4f}")
    print(f"RF + SMOTE:                 {scores_smote.mean():.4f} ± {scores_smote.std():.4f}")

else:
    print("SEVERE → Use both SMOTE + Tomek links (clean majority boundary).")

    from imblearn.combine import SMOTETomek
    from imblearn.pipeline import Pipeline as ImbPipeline

    smotetomek_rf = ImbPipeline([
        ('resample', SMOTETomek(random_state=42)),
        ('clf',      RandomForestClassifier(n_estimators=200,
                                             random_state=42, n_jobs=-1))
    ])
    scores_st = cross_val_score(smotetomek_rf, X_real, y,
                                cv=cv, scoring='f1_macro')
    print(f"RF + SMOTETomek: {scores_st.mean():.4f} ± {scores_st.std():.4f}")

# ── 5. Per-class F1 before vs after — see if weak classes improved ────────────
from sklearn.metrics import classification_report

y_pred = best_rf.predict(X_test_r)
report = classification_report(y_test_r, y_pred,
                                target_names=OB_ORDER, output_dict=True)

f1_df = pd.DataFrame({cls: report[cls] for cls in OB_ORDER}).T[['precision','recall','f1-score','support']]
f1_df = f1_df.sort_values('f1-score')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if f < 0.75 else '#fee08b' if f < 0.85 else '#1a9850'
          for f in f1_df['f1-score']]
bars = ax.barh(f1_df.index, f1_df['f1-score'], color=colors)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.axvline(0.85, color='gray', linestyle='--', label='0.85 target')
ax.set_title("Per-Class F1 Score — Best RF (Realistic Features)")
ax.set_xlabel("F1-Score")
ax.legend()
plt.tight_layout()
plt.show()

print("\nWeakest classes:")
print(f1_df[f1_df['f1-score'] < 0.85][['f1-score','support']])

- The imbalance ratio is ~1.29 (351/272), almost perfectly balanced.
- SMOTE won't help here.
- Errors concentrated in the middle classes (Normal_Weight, OW_I) are treated as a feature problem, not a sampling problem.
- Without BMI/Weight information these middle classes are hard to separate.

# Understand exactly what's getting confused

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd
import numpy as np

y_pred = best_rf.predict(X_test_r)
cm = confusion_matrix(y_test_r, y_pred)
cm_df = pd.DataFrame(cm, index=OB_ORDER, columns=OB_ORDER)

# Off-diagonal confusion pairs (most confused class pairs)
confusion_pairs = []
for i in range(7):
    for j in range(7):
        if i != j and cm[i, j] > 0:
            confusion_pairs.append({
                'true':       OB_ORDER[i],
                'predicted':  OB_ORDER[j],
                'count':      cm[i, j],
                'true_total': cm[i].sum(),
                'error_rate': round(cm[i, j] / cm[i].sum(), 3)
            })

conf_df = (pd.DataFrame(confusion_pairs)
             .sort_values('count', ascending=False)
             .head(10)
             .reset_index(drop=True))
print("Top confusion pairs:")
print(conf_df.to_string())

# Feature distributions for the confused classes only

In [ ]:
# Isolate samples where Normal_Weight and OW_I are confused
mask_true_normal  = y_test_r.values == 1   # Normal_Weight
mask_pred_ow1     = y_pred == 2            # predicted as OW_I
mask_true_ow1     = y_test_r.values == 2   # OW_I
mask_pred_normal  = y_pred == 1            # predicted as Normal_Weight

confused_idx = np.where(
    (mask_true_normal & mask_pred_ow1) |
    (mask_true_ow1    & mask_pred_normal)
)[0]

correct_idx = np.where(
    (mask_true_normal & (y_pred == 1)) |
    (mask_true_ow1    & (y_pred == 2))
)[0]

confused_df = X_test_r.iloc[confused_idx].copy()
correct_df  = X_test_r.iloc[correct_idx].copy()
confused_df['group'] = 'confused'
correct_df['group']  = 'correct'

compare_df = pd.concat([confused_df, correct_df])

# Plot feature distributions: confused vs correct
key_feats = ['Age', 'FCVC', 'FAF', 'CAEC', 'family_history_with_overweight',
             'Age_x_FCVC', 'NCP', 'CH2O']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, feat in enumerate(key_feats):
    sns.boxplot(data=compare_df, x='group', y=feat,
                palette={'confused': '#d73027', 'correct': '#1a9850'},
                ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel("")

plt.suptitle("Confused vs Correctly Classified\n(Normal_Weight ↔ OW_I)", 
             fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nConfused samples:  {len(confused_idx)}")
print(f"Correct samples:   {len(correct_idx)}")
print("\nMean feature values — confused vs correct:")
print(compare_df.groupby('group')[key_feats].mean().round(3).T)

# ordinal classification — exploit the natural class order

In [ ]:
# Convert to ordinal binary problems (rank threshold method)
# Class k → binary: "is sample above threshold k?"
# Train 6 binary classifiers, reconstruct 7-class prediction

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

scores_base = cross_val_score(best_rf, X_real, y, cv=cv, scoring='f1_macro')

def ordinal_predict(X_train, y_train, X_test, n_classes=7):
    classifiers = []
    for k in range(1, n_classes):          # thresholds 1..6
        y_bin = (y_train >= k).astype(int)
        clf = RandomForestClassifier(n_estimators=200, class_weight='balanced',
                                      random_state=42, n_jobs=-1)
        clf.fit(X_train, y_bin)
        classifiers.append(clf)

    # P(Y > k) for each threshold
    probas = np.column_stack([clf.predict_proba(X_test)[:, 1]
                               for clf in classifiers])   # (n, 6)

    # P(Y = k) = P(Y > k-1) - P(Y > k)
    p0 = 1 - probas[:, 0]                              # P(Y=0)
    p_mid = probas[:, :-1] - probas[:, 1:]             # P(Y=1..5)
    p6 = probas[:, -1]                                  # P(Y=6)
    class_probs = np.column_stack([p0, p_mid, p6])

    return np.argmax(class_probs, axis=1)

# Evaluate with CV manually
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

cv_ord = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
ord_scores = []
for train_idx, val_idx in cv_ord.split(X_real, y):
    X_tr, X_val = X_real.iloc[train_idx], X_real.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx],      y.iloc[val_idx]
    y_pred_ord = ordinal_predict(X_tr, y_tr, X_val)
    ord_scores.append(f1_score(y_val, y_pred_ord, average='macro'))

print(f"Ordinal RF F1-macro:    {np.mean(ord_scores):.4f} ± {np.std(ord_scores):.4f}")
print(f"Standard RF F1-macro:   {scores_base.mean():.4f} ± {scores_base.std():.4f}")
print(f"Improvement:            {np.mean(ord_scores) - scores_base.mean():+.4f}")

# Add BMI back — but only as a noisy proxy

In [ ]:
# WHO BMI categories as ordinal feature (0–3)
# This is less leaky than exact BMI — it's coarser and clinically meaningful
df_enc['BMI_bucket'] = pd.cut(
    df['BMI'],
    bins=[0, 18.5, 25, 30, 100],
    labels=[0, 1, 2, 3]          # under / normal / overweight / obese
).astype(int)

X_with_bucket = X_real.copy()
X_with_bucket['BMI_bucket'] = df_enc['BMI_bucket'].values

scores_bucket = cross_val_score(
    best_rf, X_with_bucket, y, cv=cv, scoring='f1_macro')

print(f"RF + BMI_bucket: {scores_bucket.mean():.4f} ± {scores_bucket.std():.4f}")
print(f"RF realistic:    {scores_base.mean():.4f} ± {scores_base.std():.4f}")
print(f"RF full (exact): {score_full.mean():.4f} ± {score_full.std():.4f}")

## Observation of the confusion
- Comparing the no-BMI, BMI-bucket, and exact-BMI runs shows how much the middle classes depend on weight information.
- A coarse bucket (4 categories) is used instead of exact BMI, so no exact weight is needed.

What this means for deployment
The BMI bucket is clinically defensible because:

Anyone can self-report approximate height/weight
A doctor/app already knows this
You're not computing BMI algorithmically from the features — you're asking "are you roughly underweight/normal/overweight/obese?"

Recommended final model: RF + BMI_bucket.

# lock in the final model and evaluate it properly

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold

# ── 1. Final feature set ──────────────────────────────────────────────────────
X_final = X_with_bucket.copy()
print("Final features:", X_final.columns.tolist())
print("Shape:", X_final.shape)

# ── 2. Tune RF on the final feature set ──────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist = {
    'n_estimators':     [200, 400, 600],
    'max_depth':        [None, 10, 20, 30],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', 0.5],
    'class_weight':     ['balanced', None],
}

rf_final_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=40, cv=cv,
    scoring='f1_macro', random_state=42, n_jobs=-1, verbose=1
)
rf_final_search.fit(X_final, y)

print(f"\nBest F1-macro (tuned): {rf_final_search.best_score_:.4f}")
print(f"Best params: {rf_final_search.best_params_}")
rf_best_final = rf_final_search.best_estimator_

# ── 3. Final holdout evaluation ───────────────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y)

rf_best_final.fit(X_tr, y_tr)
y_pred_final = rf_best_final.predict(X_te)

print("\nFinal classification report:")
print(classification_report(y_te, y_pred_final, target_names=OB_ORDER))

# ── 4. Confusion matrix ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_te, y_pred_final,
    display_labels=[o.replace('_', '\n') for o in OB_ORDER],
    cmap='Blues', colorbar=False, ax=ax
)
ax.set_title("Final Model — RF + BMI_bucket\nConfusion Matrix")
plt.tight_layout()
plt.show()

# ── 5. Per-class F1 comparison: realistic vs final ────────────────────────────
y_pred_real = best_rf.predict(X_test_r)
report_real  = classification_report(y_test_r, y_pred_real,
                                      target_names=OB_ORDER, output_dict=True)
report_final = classification_report(y_te, y_pred_final,
                                      target_names=OB_ORDER, output_dict=True)

f1_compare = pd.DataFrame({
    'Realistic (no BMI)': {c: report_real[c]['f1-score']  for c in OB_ORDER},
    'Final (BMI_bucket)': {c: report_final[c]['f1-score'] for c in OB_ORDER},
})

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(OB_ORDER))
w = 0.35
ax.bar(x - w/2, f1_compare['Realistic (no BMI)'], w,
       label='Realistic (no BMI)', color='#d73027', alpha=0.85)
ax.bar(x + w/2, f1_compare['Final (BMI_bucket)'], w,
       label='Final (BMI_bucket)',  color='#1a9850', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([o.replace('_', '\n') for o in OB_ORDER], fontsize=9)
ax.set_ylabel("F1-Score")
ax.set_ylim(0.5, 1.05)
ax.axhline(0.85, color='gray', linestyle='--', alpha=0.5, label='0.85 target')
ax.set_title("Per-Class F1: Realistic vs Final Model (BMI_bucket)")
ax.legend()
ax.bar_label(ax.containers[0], fmt='%.2f', padding=2, fontsize=8)
ax.bar_label(ax.containers[1], fmt='%.2f', padding=2, fontsize=8)
plt.tight_layout()
plt.show()

# SHAP on the final model

In [ ]:
explainer_final = shap.TreeExplainer(rf_best_final)
shap_values_final = explainer_final.shap_values(X_te)
shap_arr = np.array(shap_values_final)

# Handle shape — new SHAP returns (n_samples, n_features, n_classes)
if shap_arr.ndim == 3 and shap_arr.shape[2] == 7:
    shap_list = [shap_arr[:, :, i] for i in range(7)]
else:
    shap_list = shap_values_final   # old format already a list

# Global importance across all classes
shap.summary_plot(shap_list, X_te,
                  class_names=OB_ORDER, plot_type="bar")

# Where does BMI_bucket rank vs lifestyle features?
importance_final = pd.DataFrame({
    'feature':    X_final.columns,
    'importance': rf_best_final.feature_importances_
}).sort_values('importance', ascending=False)

print(importance_final.to_string(index=False))

# Explanation 
- BMI_bucket     0.426  ← weight signal (coarse, clinically askable)
- FCVC           0.125  ← diet quality (vegetable consumption)
- Age            0.080  ← demographic
- Gender         0.074  ← demographic  
- Age_x_FCVC     0.048  ← interaction feature I engineered — earned its place
- NCP            0.042  ← meal frequency
- TUE            0.041  ← screen time
- FAF            0.036  ← physical activity

# Three things stand out:

- family_history (0.018) ranked below NCP, TUE, FAF — completely contradicts the EDA signal. The tree is using BMI_bucket to do the heavy lifting that family_history was doing in the no-BMI model.
- FAVC (0.014) is almost noise — consistent with what SHAP showed earlier.
- All 4 MTRANS features combined = 0.012 — drop them.

# Final lean model

In [ ]:
# ── Drop near-zero importance features ───────────────────────────────────────
drop_final = ['MTRANS_Bike', 'MTRANS_Motorbike', 
              'MTRANS_Walking', 'MTRANS_Public_Transportation', 'FAVC']

X_lean = X_final.drop(columns=drop_final)
print("Lean feature set:", X_lean.columns.tolist())
print("Features dropped:", len(drop_final), 
      "| Remaining:", X_lean.shape[1])

# ── Verify dropping them doesn't hurt ────────────────────────────────────────
scores_lean = cross_val_score(
    rf_best_final, X_lean, y, cv=cv, scoring='f1_macro')
scores_full_final = cross_val_score(
    rf_best_final, X_final, y, cv=cv, scoring='f1_macro')

print(f"\nFull (17 features): {scores_full_final.mean():.4f} ± {scores_full_final.std():.4f}")
print(f"Lean (12 features): {scores_lean.mean():.4f}  ± {scores_lean.std():.4f}")
print(f"Delta:              {scores_lean.mean() - scores_full_final.mean():+.4f}")

# Save the final model

In [ ]:
import joblib
import json

# ── Retrain on full data with lean features ───────────────────────────────────
params = rf_best_final.get_params()
params['random_state'] = 42
params['n_jobs'] = -1

rf_lean_final = RandomForestClassifier(**params)
rf_lean_final.fit(X_lean, y)

# Save model + metadata
joblib.dump(rf_lean_final, 'obesity_rf_final.joblib')

# json.dump can't handle numpy types — convert params to plain Python
clean_params = {k: (int(v) if hasattr(v, 'item') else v) 
                for k, v in params.items() 
                if v is not None}

metadata = {
    'model':          'RandomForestClassifier',
    'params':         clean_params,
    'features':       X_lean.columns.tolist(),
    'target_classes': OB_ORDER,
    'label_mapping':  {k: int(v) for k, v in label_mapping.items()},
    'cv_f1_macro':    round(float(scores_lean.mean()), 4),
    'notes': [
        'BMI_bucket = pd.cut(BMI, bins=[0,18.5,25,30,100], labels=[0,1,2,3])',
        'BMI = Weight / Height^2 — coarse category only, not exact value',
        'MTRANS and FAVC dropped (importance < 0.015)',
        'Age_x_FCVC interaction feature included'
    ]
}

with open('obesity_rf_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Model saved: obesity_rf_final.joblib")
print("Metadata saved: obesity_rf_metadata.json")
print(json.dumps(metadata, indent=2))

# Inference function — how to use this model on new data

In [ ]:
def predict_obesity(gender, age, family_history, fcvc, ncp,
                    caec, ch2o, faf, tue, calc, bmi_bucket):
    """
    gender:         0=Female, 1=Male
    age:            float (years)
    family_history: 0=no, 1=yes
    fcvc:           1-3 (vegetable consumption frequency)
    ncp:            1-4 (number of main meals)
    caec:           0=no, 1=Sometimes, 2=Frequently, 3=Always
    ch2o:           1-3 (daily water intake)
    faf:            0-3 (physical activity frequency)
    tue:            0-2 (technology use hours)
    calc:           0=no, 1=Sometimes, 2=Frequently, 3=Always
    bmi_bucket:     0=Underweight(<18.5), 1=Normal(18.5-25),
                    2=Overweight(25-30),  3=Obese(>30)
    """
    features = pd.DataFrame([{
        'Gender':                           gender,
        'Age':                              age,
        'family_history_with_overweight':   family_history,
        'FCVC':                             fcvc,
        'NCP':                              ncp,
        'CAEC':                             caec,
        'CH2O':                             ch2o,
        'FAF':                              faf,
        'TUE':                              tue,
        'CALC':                             calc,
        'Age_x_FCVC':                       age * fcvc,
        'BMI_bucket':                       bmi_bucket,
    }])

    pred       = rf_lean_final.predict(features)[0]
    proba      = rf_lean_final.predict_proba(features)[0]
    confidence = proba.max()

    print(f"Predicted class:  {OB_ORDER[pred]}")
    print(f"Confidence:       {confidence:.1%}")
    print(f"\nFull probability distribution:")
    for cls, p in zip(OB_ORDER, proba):
        bar = '█' * int(p * 30)
        print(f"  {cls:25s} {p:.3f} {bar}")

    return pred, proba

# ── Test on a sample ──────────────────────────────────────────────────────────
predict_obesity(
    gender=1, age=25, family_history=1,
    fcvc=2, ncp=3, caec=1, ch2o=2,
    faf=1, tue=1, calc=1,
    bmi_bucket=2   # overweight range
)

# Making a better confidence score approach

### Pre-compute components for every sample (the weights)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# Scale the lean feature set
scaler = StandardScaler()
X_lean_scaled = scaler.fit_transform(X_lean)

# Class centroids in scaled space
centroids = {}
demographic_profiles = {}

for class_idx, class_name in enumerate(OB_ORDER):
    mask       = (y == class_idx)
    class_data = X_lean_scaled[mask]
    centroids[class_idx] = class_data.mean(axis=0)

    raw_class = X_lean[mask]
    demographic_profiles[class_idx] = {
        'age_mean':      raw_class['Age'].mean(),
        'age_std':       raw_class['Age'].std(),
        'gender_rate':   raw_class['Gender'].mean(),
        'fam_hist_rate': raw_class['family_history_with_overweight'].mean(),
        'fcvc_mean':     raw_class['FCVC'].mean(),
    }

print("Centroids ready for", len(centroids), "classes")

### Loop through all samples and store x, y, z, m

In [ ]:
print("Pre-computing components for all samples...")

all_components = []
all_correct    = []

for i in range(len(X_lean)):
    row      = X_lean.iloc[[i]]
    proba    = rf_lean_final.predict_proba(row)[0]
    pred_idx = proba.argmax()
    true_idx = int(y.iloc[i])

    top3_idx  = proba.argsort()[::-1][:3]
    runner_up = [(top3_idx[1], proba[top3_idx[1]]),
                 (top3_idx[2], proba[top3_idx[2]])]

    # x — cosine similarity vs predicted class centroid
    input_scaled  = scaler.transform(row)
    centroid_pred = centroids[pred_idx].reshape(1, -1)
    x_score       = float(max(0, cosine_similarity(
                        input_scaled, centroid_pred)[0, 0]))

    # z — confusion risk from runner-up classes
    z_parts = []
    for ru_idx, ru_proba in runner_up:
        cos_ru = float(max(0, cosine_similarity(
            input_scaled, centroids[ru_idx].reshape(1, -1))[0, 0]))
        z_parts.append(ru_proba * cos_ru)
    z_score = np.mean(z_parts)

    # m — demographic alignment
    inp          = row.iloc[0]
    profile      = demographic_profiles[pred_idx]
    age_z        = abs(inp['Age'] - profile['age_mean']) / (profile['age_std'] + 1e-6)
    age_match    = max(0.0, 1 - age_z / 3)
    gender_match = (inp['Gender'] * profile['gender_rate'] +
                    (1 - inp['Gender']) * (1 - profile['gender_rate']))
    fam_match    = float(inp['family_history_with_overweight'] ==
                         round(profile['fam_hist_rate']))
    m_score      = 0.4 * age_match + 0.4 * gender_match + 0.2 * fam_match

    all_components.append({
        'x': x_score,
        'y': float(proba[pred_idx]),
        'z': z_score,
        'm': m_score,
    })
    all_correct.append(int(pred_idx == true_idx))

comp_df = pd.DataFrame(all_components)
correct = np.array(all_correct)

print(f"Done. Correct: {correct.sum()} | Wrong: {(1-correct).sum()}")
print(comp_df.describe().round(3))

###  Define the objective function

In [ ]:
def score_with_weights(weights, comp_df):
    a, b, c, d = weights
    scores = np.clip(
        a * comp_df['x'].values +
        b * comp_df['y'].values -
        c * comp_df['z'].values +
        d * comp_df['m'].values,
        0.0, 1.0
    )
    return scores

def objective(weights, comp_df, correct, alpha=0.7, beta=0.3):
    a, b, c, d = weights

    # All weights must be positive
    if any(w < 0 for w in [a, b, c, d]):
        return 999.0

    scores         = score_with_weights(weights, comp_df)
    correct_scores = scores[correct == 1]
    wrong_scores   = scores[correct == 0]

    # Guard against empty arrays
    if len(correct_scores) == 0 or len(wrong_scores) == 0:
        return 999.0

    separation       = correct_scores.mean() - wrong_scores.mean()
    false_confidence = (wrong_scores > 0.5).mean()
    true_confidence  = (correct_scores > 0.7).mean()

    # Guard against NaN in any component
    if np.isnan(separation) or np.isnan(false_confidence) or np.isnan(true_confidence):
        return 999.0

    loss = (-alpha * separation
            - beta * true_confidence
            + (1 - alpha - beta) * false_confidence)
    return float(loss)

# Quick sanity check before running grid
test_loss = objective((0.35, 0.40, 0.15, 0.10), comp_df, correct)
print(f"Sanity check loss: {test_loss:.4f}  ← should be a real number, not NaN")
print(f"comp_df shape: {comp_df.shape}")
print(f"correct: {correct.sum()} correct, {(1-correct).sum()} wrong")
print(comp_df.describe().round(3))

### Grid search (broad sweep)

In [ ]:
from itertools import product

print("Running grid search...")

a_vals = np.arange(0.10, 0.55, 0.05)
b_vals = np.arange(0.20, 0.65, 0.05)
c_vals = np.arange(0.05, 0.35, 0.05)
d_vals = np.arange(0.05, 0.30, 0.05)

best_loss    = 999.0
best_weights = None
results      = []

for a, b, c, d in product(a_vals, b_vals, c_vals, d_vals):
    loss = objective((a, b, c, d), comp_df, correct)
    results.append({'a': a, 'b': b, 'c': c, 'd': d, 'loss': loss})
    if loss < best_loss:
        best_loss    = loss
        best_weights = (a, b, c, d)

results_df = pd.DataFrame(results).sort_values('loss').reset_index(drop=True)
print("Top 10 grid results:")
print(results_df.head(10).round(4).to_string(index=False))
print(f"\nBest grid weights: a={best_weights[0]:.2f}, b={best_weights[1]:.2f}, "
      f"c={best_weights[2]:.2f}, d={best_weights[3]:.2f}  |  loss={best_loss:.4f}")

# model learnt everything lol

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics.pairwise import cosine_similarity

# ── Get out-of-fold predictions (never seen during training) ──────────────────
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_proba = cross_val_predict(rf_lean_final, X_lean, y,
                               cv=cv5, method='predict_proba')
oof_pred  = oof_proba.argmax(axis=1)
oof_correct = (oof_pred == y.values).astype(int)

print(f"OOF correct: {oof_correct.sum()} | OOF wrong: {(1-oof_correct).sum()}")
# Now you should see actual wrong predictions

In [ ]:
# ── Recompute components using OOF probabilities ──────────────────────────────
print("Computing components from OOF predictions...")

all_components = []

for i in range(len(X_lean)):
    row      = X_lean.iloc[[i]]
    proba    = oof_proba[i]           # OOF proba — not seen during training
    pred_idx = proba.argmax()

    top3_idx  = proba.argsort()[::-1][:3]
    runner_up = [(top3_idx[1], proba[top3_idx[1]]),
                 (top3_idx[2], proba[top3_idx[2]])]

    # x — cosine similarity vs predicted class centroid
    input_scaled  = scaler.transform(row)
    x_score       = float(max(0, cosine_similarity(
                        input_scaled,
                        centroids[pred_idx].reshape(1, -1))[0, 0]))

    # z — confusion risk from runner-ups
    z_parts = []
    for ru_idx, ru_proba in runner_up:
        cos_ru = float(max(0, cosine_similarity(
            input_scaled,
            centroids[ru_idx].reshape(1, -1))[0, 0]))
        z_parts.append(ru_proba * cos_ru)
    z_score = np.mean(z_parts)

    # m — demographic alignment
    inp          = row.iloc[0]
    profile      = demographic_profiles[pred_idx]
    age_z        = abs(inp['Age'] - profile['age_mean']) / (profile['age_std'] + 1e-6)
    age_match    = max(0.0, 1 - age_z / 3)
    gender_match = (inp['Gender'] * profile['gender_rate'] +
                    (1 - inp['Gender']) * (1 - profile['gender_rate']))
    fam_match    = float(inp['family_history_with_overweight'] ==
                         round(profile['fam_hist_rate']))
    m_score      = 0.4 * age_match + 0.4 * gender_match + 0.2 * fam_match

    all_components.append({
        'x': x_score,
        'y': float(proba[pred_idx]),
        'z': z_score,
        'm': m_score,
    })

comp_df = pd.DataFrame(all_components)
correct = oof_correct   # use OOF correct, not training correct

print(f"Done. Correct: {correct.sum()} | Wrong: {(1-correct).sum()}")
print(comp_df.describe().round(3))

### Grid search (broad sweep)

In [ ]:
from itertools import product

print("Running grid search...")

a_vals = np.arange(0.10, 0.55, 0.05)
b_vals = np.arange(0.20, 0.65, 0.05)
c_vals = np.arange(0.05, 0.35, 0.05)
d_vals = np.arange(0.05, 0.30, 0.05)

best_loss    = 999.0
best_weights = None
results      = []

for a, b, c, d in product(a_vals, b_vals, c_vals, d_vals):
    loss = objective((a, b, c, d), comp_df, correct)
    results.append({'a': a, 'b': b, 'c': c, 'd': d, 'loss': loss})
    if loss < best_loss:
        best_loss    = loss
        best_weights = (a, b, c, d)

results_df = pd.DataFrame(results).sort_values('loss').reset_index(drop=True)
print("Top 10 grid results:")
print(results_df.head(10).round(4).to_string(index=False))
print(f"\nBest grid weights: a={best_weights[0]:.2f}, b={best_weights[1]:.2f}, "
      f"c={best_weights[2]:.2f}, d={best_weights[3]:.2f}  |  loss={best_loss:.4f}")

###  Fine-tune with Nelder-Mead

In [ ]:
from scipy.optimize import minimize

result_opt = minimize(
    objective,
    x0=list(best_weights),
    args=(comp_df, correct),
    method='Nelder-Mead',
    options={'xatol': 1e-5, 'fatol': 1e-5, 'maxiter': 50000}
)

opt_weights = tuple(np.clip(result_opt.x, 0.01, 0.99))
print(f"Optimized weights:")
print(f"  a (cosine sim):     {opt_weights[0]:.4f}")
print(f"  b (model proba):    {opt_weights[1]:.4f}")
print(f"  c (confusion risk): {opt_weights[2]:.4f}")
print(f"  d (demographic):    {opt_weights[3]:.4f}")
print(f"  loss:               {result_opt.fun:.4f}")

### Evaluate all three weight sets

In [ ]:
def evaluate_weights(weights, comp_df, correct, label):
    scores         = score_with_weights(weights, comp_df)
    correct_scores = scores[correct == 1]
    wrong_scores   = scores[correct == 0]
    gap            = correct_scores.mean() - wrong_scores.mean()

    print(f"\n── {label} ──")
    print(f"  weights:          a={weights[0]:.3f} b={weights[1]:.3f} "
          f"c={weights[2]:.3f} d={weights[3]:.3f}")
    print(f"  correct mean:     {correct_scores.mean():.4f} "
          f"| >0.7: {(correct_scores > 0.7).mean():.1%}")
    print(f"  wrong mean:       {wrong_scores.mean():.4f}  "
          f"| >0.5: {(wrong_scores > 0.5).mean():.1%}  ← want LOW")
    print(f"  separation gap:   {gap:.4f}  ← want HIGH")
    return scores, gap

s_default, g_default = evaluate_weights(
    (0.35, 0.40, 0.15, 0.10), comp_df, correct, "Default")
s_grid,    g_grid    = evaluate_weights(
    best_weights,              comp_df, correct, "Grid best")
s_opt,     g_opt     = evaluate_weights(
    opt_weights,               comp_df, correct, "Optimized")

### Plot distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (label, scores, gap) in zip(axes, [
    ("Default",   s_default, g_default),
    ("Grid best", s_grid,    g_grid),
    ("Optimized", s_opt,     g_opt),
]):
    ax.hist(scores[correct == 1], bins=25, alpha=0.6,
            color='#1a9850', label='Correct', density=True)
    ax.hist(scores[correct == 0], bins=25, alpha=0.6,
            color='#d73027', label='Wrong',   density=True)
    ax.axvline(0.5, color='black', linestyle='--', alpha=0.5, label='0.5 threshold')
    ax.set_title(f"{label}\ngap = {gap:.4f}")
    ax.set_xlabel("Confidence Score")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.suptitle("Confidence Score Distribution — Correct vs Wrong", fontsize=13)
plt.tight_layout()
plt.show()

###  Lock in the best weights

In [ ]:
# Auto-select the weight set with the highest separation gap
candidates = {
    'default':   ((0.35, 0.40, 0.15, 0.10), g_default),
    'grid':      (best_weights,               g_grid),
    'optimized': (opt_weights,                g_opt),
}
best_name    = max(candidates, key=lambda k: candidates[k][1])
FINAL_WEIGHTS = candidates[best_name][0]

print(f"✅ Selected: {best_name}")
print(f"   a={FINAL_WEIGHTS[0]:.4f}  b={FINAL_WEIGHTS[1]:.4f}  "
      f"c={FINAL_WEIGHTS[2]:.4f}  d={FINAL_WEIGHTS[3]:.4f}")
print(f"   Separation gap: {candidates[best_name][1]:.4f}")

# final predict function

In [ ]:
def predict_obesity_final(gender, age, family_history, fcvc, ncp,
                          caec, ch2o, faf, tue, calc, bmi_bucket):

    features = pd.DataFrame([{
        'Gender':                           gender,
        'Age':                              age,
        'family_history_with_overweight':   family_history,
        'FCVC':                             fcvc,
        'NCP':                              ncp,
        'CAEC':                             caec,
        'CH2O':                             ch2o,
        'FAF':                              faf,
        'TUE':                              tue,
        'CALC':                             calc,
        'Age_x_FCVC':                       age * fcvc,
        'BMI_bucket':                       bmi_bucket,
    }])

    # Components
    proba        = rf_lean_final.predict_proba(features)[0]
    pred_idx     = proba.argmax()
    top3_idx     = proba.argsort()[::-1][:3]
    runner_up    = [(top3_idx[1], proba[top3_idx[1]]),
                    (top3_idx[2], proba[top3_idx[2]])]

    input_scaled = scaler.transform(features)
    x_score      = float(max(0, cosine_similarity(
                       input_scaled, centroids[pred_idx].reshape(1,-1))[0,0]))

    z_parts = []
    for ru_idx, ru_proba in runner_up:
        cos_ru = float(max(0, cosine_similarity(
            input_scaled, centroids[ru_idx].reshape(1,-1))[0,0]))
        z_parts.append(ru_proba * cos_ru)
    z_score = np.mean(z_parts)

    inp          = features.iloc[0]
    profile      = demographic_profiles[pred_idx]
    age_z        = abs(inp['Age'] - profile['age_mean']) / (profile['age_std'] + 1e-6)
    age_match    = max(0.0, 1 - age_z / 3)
    gender_match = (inp['Gender'] * profile['gender_rate'] +
                    (1 - inp['Gender']) * (1 - profile['gender_rate']))
    fam_match    = float(inp['family_history_with_overweight'] ==
                         round(profile['fam_hist_rate']))
    m_score      = 0.4 * age_match + 0.4 * gender_match + 0.2 * fam_match

    comp = pd.DataFrame([{'x': x_score,
                          'y': float(proba[pred_idx]),
                          'z': z_score,
                          'm': m_score}])

    confidence = float(score_with_weights(FINAL_WEIGHTS, comp)[0])

    # ── Pretty print ──────────────────────────────────────────────────────────
    filled = int(confidence * 40)
    bar    = '█' * filled + '░' * (40 - filled)

    print(f"┌{'─'*52}┐")
    print(f"│  Predicted : {OB_ORDER[pred_idx]:35s}│")
    print(f"│  Confidence: [{bar}] {confidence*100:4.1f}%  │")
    print(f"└{'─'*52}┘")
    print(f"\n  Score breakdown:")
    print(f"    x  cosine sim (predicted class):  {x_score:.4f}")
    print(f"    y  model probability:              {proba[pred_idx]:.4f}")
    print(f"    z  confusion risk (runner-ups):   {z_score:.4f}")
    print(f"    m  demographic alignment:          {m_score:.4f}")
    print(f"\n  Runner-up classes:")
    for ru_idx, ru_p in runner_up:
        b2 = '█' * int(ru_p * 30)
        print(f"    {OB_ORDER[ru_idx]:25s} {ru_p:.3f} {b2}")
    print(f"\n  Full distribution:")
    for i, (cls, p) in enumerate(zip(OB_ORDER, proba)):
        b3 = '█' * int(p * 30)
        print(f"    {cls:25s} {p:.3f} {b3}")

    return {
        'predicted':   OB_ORDER[pred_idx],
        'confidence':  round(confidence, 4),
        'components':  {'x': x_score, 'y': float(proba[pred_idx]),
                        'z': z_score,  'm': m_score},
        'runner_up':   [(OB_ORDER[i], round(p,4)) for i,p in runner_up],
        'all_probas':  {OB_ORDER[i]: round(float(p),4) for i,p in enumerate(proba)},
        'weights_used': FINAL_WEIGHTS,
    }

# ── Test ──────────────────────────────────────────────────────────────────────
predict_obesity_final(
    gender=1, age=25, family_history=1,
    fcvc=2, ncp=3, caec=1, ch2o=2,
    faf=1, tue=1, calc=1,
    bmi_bucket=2
)

# Required inputs

In [ ]:
# What the user/API must provide:
{
    "gender":           1,      # 0 = Female, 1 = Male
    "age":              25.0,   # float, years
    "family_history":   1,      # 0 = no, 1 = yes
    "fcvc":             2.0,    # 1–3  (vegetable consumption: 1=never, 2=sometimes, 3=always)
    "ncp":              3.0,    # 1–4  (main meals per day)
    "caec":             1,      # 0=no, 1=sometimes, 2=frequently, 3=always (snacking)
    "ch2o":             2.0,    # 1–3  (water intake: 1=<1L, 2=1-2L, 3=>2L)
    "faf":              1.0,    # 0–3  (physical activity days/week)
    "tue":              1.0,    # 0–2  (screen time hours: 0=0-2h, 1=3-5h, 2=5h+)
    "calc":             1,      # 0=no, 1=sometimes, 2=frequently, 3=always (alcohol)
    "bmi_bucket":       2       # 0=underweight(<18.5), 1=normal(18.5-25),
                                # 2=overweight(25-30), 3=obese(>30)
}